# Notebook 2: Model Training & Evaluation (TF-IDF + SVM vs IndoBERT)

Notebook ini berfokus murni pada komputasi berat, yaitu **Pelatihan Model**.

**Spesifikasi:**
- Membutuhkan GPU (Sangat disarankan T4 GPU di Colab).
- Mengunci seed di angka 42 untuk reproduksibilitas akademis.
- Model Baseline: TF-IDF + Support Vector Machine (Linear Kernel).
- Model Utama: IndoBERT Base (`indobenchmark/indobert-base-p1`).
- Menggunakan konfigurasi efisien: **Batch Size 16** dan **FP16=True** untuk menghindari *CUDA Out of Memory (OOM)* pada GPU T4 (15GB VRAM).

**Kelompok 14:**
- Azka Nabihan Hilmy (2306250541)
- Rivi Yasha Hafizhan (230625035)
- M. Arya Wiandra Utomo (2306218295)

### Instalasi Dependensi
Menginstal pustaka Transformer dari Hugging Face dan Scikit-Learn.

In [1]:
# Install Hugging Face Transformers & Libraries
!pip install transformers datasets evaluate accelerate scikit-learn matplotlib seaborn -q
print("[INFO] Instalasi dependensi selesai.")

[INFO] Instalasi dependensi selesai.


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'c:\\Pribadi\\Kuliah\\Akademis\\Semester 6\\AI\\Finpro\\.venv\\Lib\\site-packages\\datasets\\packaged_modules\\pdffolder\\pdffolder.py'
Check the permissions.



### Pengaturan Seed (Reproduksibilitas)
Sangat kritis agar hasil eksperimen Machine Learning konstan setiap kali dijalankan.

In [3]:
import os
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print("Seed = 42")

Seed = 42


### Import Library ML

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import evaluate

c:\Pribadi\Kuliah\Akademis\Semester 6\AI\Finpro\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Load Dataset Final
Membaca dataset hasil dari ETL (Notebook 1).

In [ ]:
try:
    df = pd.read_csv('dataset_gabungan_final.csv')
    print(f"erhasil memuat data: {len(df)} baris.")
    print(df.head())
except FileNotFoundError:
    print("File 'dataset_gabungan_final.csv' tidak ditemukan!")
    print("Pastikan Anda telah menjalankan Notebook 1 dan file tersebut ada di direktori ini.")

erhasil memuat data: 12970 baris.
                                                text    label
0  israel bombardir gaza mesin uang orang terkaya...   Netral
1  isu ketahanan pangan bumn ini bikin ekosistem ...   Netral
2                                     saham bca nad🥰  Positif
3  29 tahun masuk daftar orang terkaya rahasia bi...   Netral
4  selain bi rate ini 5 jurus bi jaga ekonomi ind...  Positif


### Data Split & Label Encoding
Ubah label teks menjadi angka numerik (0, 1, 2) dan bagi menjadi data *train* (80%) dan *test* (20%).

In [ ]:
if 'df' in locals():
    # Label Encoding
    le = LabelEncoder()
    df['label_id'] = le.fit_transform(df['label'])
    
    # Simpan mapping kelas
    label_names = list(le.classes_)
    print(f"Mapping Label: {dict(enumerate(label_names))}")
    
    # Train-Test Split (Stratified 80:20)
    train_texts, test_texts, train_labels, test_labels = train_test_split(
        df['text'].tolist(), df['label_id'].tolist(),
        test_size=0.2, random_state=42, stratify=df['label_id']
    )
    
    print(f"Data Train: {len(train_texts)} sampel")
    print(f"Data Test: {len(test_texts)} sampel")

Mapping Label: {0: 'Negatif', 1: 'Netral', 2: 'Positif'}
Data Train: 10376 sampel
Data Test: 2594 sampel


### Model Baseline (TF-IDF + SVM)
Sebagai perbandingan komparatif dengan IndoBERT.

In [ ]:
# Baseline TF-IDF + SVM
print("========== Training SVM Baseline ==========")
# Ekstraksi Fitur (Unigram & Bigram)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)
X_train_tfidf = vectorizer.fit_transform(train_texts)
X_test_tfidf = vectorizer.transform(test_texts)

# Train SVC Linear Kernel
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_tfidf, train_labels)

# Evaluasi SVM
svm_preds = svm_model.predict(X_test_tfidf)
print("\n--- Classification Report (SVM) ---")
print(classification_report(test_labels, svm_preds, target_names=label_names))

========== Training SVM Baseline ==========

--- Classification Report (SVM) ---
              precision    recall  f1-score   support

     Negatif       0.77      0.75      0.76       668
      Netral       0.80      0.81      0.80      1004
     Positif       0.81      0.81      0.81       922

    accuracy                           0.79      2594
   macro avg       0.79      0.79      0.79      2594
weighted avg       0.79      0.79      0.79      2594



### Penyiapan IndoBERT (Tokenisasi & Dataset)
Membungkus teks ke dalam format Dataset PyTorch yang bisa dikonsumsi oleh Hugging Face Trainer.

In [ ]:
model_name = "indobenchmark/indobert-base-p1"
print(f"Memuat tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

# PyTorch Dataset Wrapper
class SentimentDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

# Tokenisasi (Max length dibatasi ke 128 untuk efisiensi memori, cukup untuk teks pendek/tweet)
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

train_dataset = SentimentDataset(train_encodings, train_labels)
test_dataset = SentimentDataset(test_encodings, test_labels)
print("Dataset PyTorch berhasil dibuat!")

Memuat tokenizer: indobenchmark/indobert-base-p1


✅ Dataset PyTorch berhasil dibuat!


### Konfigurasi IndoBERT Model
Mendefinisikan fungsi metrik komputasi dan memanggil *pre-trained weights* IndoBERT.

In [ ]:
import numpy as np
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": f1}

print("Memuat bobot pre-trained IndoBERT...")
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=len(label_names),
    id2label={i: l for i, l in enumerate(label_names)},
    label2id={l: i for i, l in enumerate(label_names)}
)

Memuat bobot pre-trained IndoBERT...


[transformers] You passed `num_labels=3` which is incompatible to the `id2label` map of length `5`.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 33523.44it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Sel 9: Training IndoBERT (Batch=16, FP16)
Parameter krusial: `per_device_train_batch_size=16` dan `fp16=True` sangat penting untuk menghindari VRAM GPU meledak (*Out of Memory*).

In [ ]:
# Sel 9: Training Arguments & Trainer Execution

training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,              # 3 epoch cukup untuk fine-tuning dataset teks pendek
    per_device_train_batch_size=16,  # Sesuai permintaan (sweet spot T4 GPU)
    per_device_eval_batch_size=16,
    warmup_steps=100,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,                       # Mixed precision untuk hemat VRAM
    seed=42,
    report_to="none"                 # Disable wandb dsb
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

print("========== Memulai Training IndoBERT ==========")
trainer.train()
print("Training selesai!")

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


========== Memulai Training IndoBERT ==========


c:\Pribadi\Kuliah\Akademis\Semester 6\AI\Finpro\.venv\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

### Sel 10: Evaluasi IndoBERT (Confusion Matrix)
Mencetak *classification report* akhir dan memplot *confusion matrix*.

In [ ]:
# Sel 10: Evaluasi Akhir & Visualisasi

print("========== Evaluasi IndoBERT pada Data Test ==========")
predictions, labels, _ = trainer.predict(test_dataset)
bert_preds = np.argmax(predictions, axis=-1)

print("\n--- Classification Report (IndoBERT) ---")
print(classification_report(labels, bert_preds, target_names=label_names))

# Plot Confusion Matrix
cm = confusion_matrix(labels, bert_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=label_names, yticklabels=label_names)
plt.title('Confusion Matrix - IndoBERT')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()